# Create Master Dataset

This notebook loads several raw preschool datasets, extracts a centre code from location HTML, merges centre, licence, location, and service data, cleans invalid identifiers, and exports a single master JSON dataset for downstream use.

In [1]:
!pip install pandas
!pip install geopandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


This cell installs the required Python packages for the notebook environment, ensuring pandas and geopandas are available before loading the dataset files.

In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# Locate the repository root whether Jupyter started at the root or notebook folder.
start_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (start_dir, *start_dir.parents) if (path / "SystemCode").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the KinderCompass repository root")

# Load the source datasets.
raw_data_dir = repo_root / "SystemCode/data/raw"
df_centres_path = raw_data_dir / "ListingofCentres.csv"
df_centres = pd.read_csv(df_centres_path, dtype={"postal_code": "string"})

# Give every source record a stable internal identifier. ECDA rows without a
# centre_code retain their unique tp_code instead of being discarded.
valid_centre_code = df_centres["centre_code"].notna() & df_centres["centre_code"].ne("na")
valid_tp_code = df_centres["tp_code"].notna() & df_centres["tp_code"].ne("na")
df_centres["school_id"] = pd.NA
df_centres.loc[valid_centre_code, "school_id"] = "CENTRE:" + df_centres.loc[valid_centre_code, "centre_code"]
df_centres.loc[~valid_centre_code & valid_tp_code, "school_id"] = "TP:" + df_centres.loc[~valid_centre_code & valid_tp_code, "tp_code"]
df_centres["identifier_type"] = valid_centre_code.map({True: "centre_code", False: "tp_code"})
if df_centres["school_id"].isna().any():
    raise ValueError("Every centre must have either a centre_code or tp_code")
if not df_centres["school_id"].is_unique:
    raise ValueError("school_id must be unique before enrichment")
df_licences_path = raw_data_dir / "ListingofCentresLicenceHistory.csv"
df_licences = pd.read_csv(df_licences_path)

# Load the geojson file using GeoPandas
df_locations_path = raw_data_dir / "PreSchoolsLocation.geojson"
df_locations = gpd.read_file(df_locations_path)

# Assign each preschool coordinate to its official URA planning area.
planning_areas_path = raw_data_dir / "MasterPlan2025PlanningArea.geojson"
planning_areas = gpd.read_file(planning_areas_path).to_crs(df_locations.crs)
df_locations = gpd.sjoin(
    df_locations,
    planning_areas[["PLN_AREA_N", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])
df_locations = df_locations.rename(columns={"PLN_AREA_N": "town"})

# Extract the centre code from the HTML description
df_locations['centre_code'] = df_locations['Description'].str.extract(r'<th>CENTRE_CODE</th>\s*<td>(.*?)</td>')

print("Centres columns:", df_locations.columns.tolist())

# Merge enrichment only for assigned centre codes. Filtering the right-hand
# tables prevents the literal placeholder 'na' from behaving like a join key.
df_locations = df_locations[df_locations["centre_code"].notna() & df_locations["centre_code"].ne("na")]
df_locations = df_locations.drop_duplicates(subset=["centre_code"], keep="last")
df_licences = df_licences[df_licences["centre_code"].notna() & df_licences["centre_code"].ne("na")]
df_licences = df_licences.sort_values("license_issue_date").drop_duplicates(subset=["centre_code"], keep="last")
df_step_1 = pd.merge(df_centres, df_locations, on="centre_code", how="left", validate="many_to_one")
df_main = pd.merge(df_step_1, df_licences, on="centre_code", how="left", validate="many_to_one")

# Load the services dataset
df_services_path = raw_data_dir / "ListingofCentreServices.csv"
df_services = pd.read_csv(df_services_path)
df_services["fees"] = pd.to_numeric(df_services["fees"], errors="coerce")
df_services = df_services[df_services["centre_code"].notna() & df_services["centre_code"].ne("na")]

# Build derived preschool-level metadata from services
df_service_summary = (
    df_services.groupby("centre_code")
    .agg(
        base_fee=("fees", "min"),
        care_levels=("levels_offered", lambda s: sorted(set(s.dropna())))
    )
    .reset_index()
)

df_services_json = (
    df_services.groupby("centre_code")[["levels_offered", "type_of_service", "type_of_citizenship", "fees"]]
    .apply(lambda x: x.to_dict(orient="records"))
    .reset_index(name="services_menu")
)

print(df_service_summary.head(10))
print(df_services_json.head(10))


Centres columns: ['Name', 'Description', 'geometry', 'centre_code']
  centre_code  base_fee                                        care_levels
0      EB0001     610.0  [Kindergarten 1 (5 yrs old), Kindergarten 2 (6...
1      EB0002     610.0  [Kindergarten 1 (5 yrs old), Kindergarten 2 (6...
2      EB0003     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
3      EB0004     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
4      EB0005     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
5      EB0006     610.0  [Kindergarten 1 (5 yrs old), Kindergarten 2 (6...
6      EB0007     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
7      EB0008     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
8      EB0009     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
9      EB0010     610.0  [Infant (2 to 18 mths), Kindergarten 1 (5 yrs ...
  centre_code                                      services_menu
0      EB0001  [{'levels_offered': 'Kinder

This code groups service rows by `centre_code` and converts the selected columns into a nested list of dictionaries for each preschool. `reset_index(name='services_menu')` produces a table keyed by `centre_code` with one `services_menu` payload per centre.

The final dataset supports these preschool-level properties:

- `services_menu`: raw service records for each centre, including `levels_offered`, `type_of_service`, `type_of_citizenship`, and `fees`.
- `base_fee`: inferred as the minimum `fees` across all service rows for the same centre.
- `care_levels`: inferred as the sorted, deduplicated list of `levels_offered` values for the same centre.
- `operator_scheme`: copied directly from the `scheme_type` field in `ListingofCentres.csv`.
- `philosophy`: inferred heuristically from the centre name:
- `pedagogy`: inferred heuristically from the centre name, using the same keyword mapping as philosophy
  - contains `montessori` → `Montessori`
  - contains `bilingual` → `Bilingual`
  - contains `reggio` → `Reggio Emilia`
  - contains `play` → `Play-based`
  - otherwise → `General`

Once the services data is nested in this format, the next step is to attach it to the other datasets.

To combine two DataFrames in Pandas, we use the `pd.merge()` function.

In [3]:
# Merge the main table and the grouped services table together
df_combined = pd.merge(df_main, df_services_json, on="centre_code", how="left")
df_combined = pd.merge(df_combined, df_service_summary, on="centre_code", how="left")

# Store the postal code as a number.
df_combined["postal_code"] = pd.to_numeric(df_combined["postal_code"], errors="coerce").astype("Int64")

# Derive operator scheme and preschool philosophy/pedagogy metadata
df_combined["operator_scheme"] = df_combined["scheme_type"]

def infer_philosophy(name):
    text = str(name or "").lower()
    if "montessori" in text:
        return "Montessori"
    if "bilingual" in text:
        return "Bilingual"
    if "reggio" in text:
        return "Reggio Emilia"
    if "play" in text:
        return "Play-based"
    return "General"

def infer_pedagogy(name):
    text = str(name or "").lower()
    if "montessori" in text:
        return "Montessori"
    if "bilingual" in text:
        return "Bilingual"
    if "reggio" in text:
        return "Reggio Emilia"
    if "play" in text:
        return "Play-based"
    return "General"

df_combined["philosophy"] = df_combined["centre_name_x"].apply(infer_philosophy)
df_combined["pedagogy"] = df_combined["centre_name_x"].apply(infer_pedagogy)

# Preserve the complete ECDA catalogue and describe enrichment coverage rather
# than deleting records that cannot support a particular feature.
vacancy_columns = [column for column in df_combined.columns if "_vacancy_" in column]
df_combined["has_location"] = df_combined["geometry"].notna()
df_combined["has_fee_data"] = df_combined["base_fee"].notna()
df_combined["has_licence_data"] = df_combined["license_issue_date"].notna()
df_combined["has_vacancy_data"] = df_combined[vacancy_columns].notna().any(axis=1)
if not df_combined["school_id"].is_unique:
    raise ValueError("Enrichment produced duplicate school_id values")

print(df_combined.info())


<class 'pandas.DataFrame'>
RangeIndex: 2091 entries, 0 to 2090
Data columns (total 80 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   tp_code                       2091 non-null   str     
 1   centre_code                   2091 non-null   str     
 2   centre_name_x                 2091 non-null   str     
 3   organisation_code             2091 non-null   str     
 4   organisation_description      2091 non-null   str     
 5   service_model                 2091 non-null   str     
 6   centre_contact_no             2091 non-null   int64   
 7   centre_email_address          2091 non-null   str     
 8   centre_address                2091 non-null   str     
 9   postal_code                   2091 non-null   int64   
 10  centre_website                2091 non-null   str     
 11  infant_vacancy_current_month  2091 non-null   str     
 12  infant_vacancy_next_month     2091 non-null   str     
 13 

Here is a quick breakdown of those parameters: on='centre_code': This tells Pandas to use the unique Centre Code as the anchor to match the rows together. how='left': This ensures that every single preschool from your df_main list is kept in the final table, even if it happens to be missing fee data in the services file.

With this step, nested JSON aggregation is complete! We now have a single, clean DataFrame (df_combined) containing all the locations, licence histories, and a bundled menu of fees and services for every preschool.  Now that all this data is perfectly prepared in Python, the pipeline will need to actually read it. 

Next is to export this final df_combined table so Neo4j Knowledge Graph and Rules Engine can load it

In [4]:
print(df_combined.head())

  tp_code centre_code                                      centre_name_x  \
0      na      ST0280  PCF Sparkletots Preschool @ Sengkang North Blk...   
1      na      ST0280  PCF Sparkletots Preschool @ Sengkang North Blk...   
2      na      ST0280  PCF Sparkletots Preschool @ Sengkang North Blk...   
3      na      ST0344  PCF Sparkletots Preschool @ Admiralty Blk 687B...   
4      na      ST0344  PCF Sparkletots Preschool @ Admiralty Blk 687B...   

  organisation_code           organisation_description service_model  \
0                ST  PCF Sparkletots Preschool Limited            DS   
1                ST  PCF Sparkletots Preschool Limited            DS   
2                ST  PCF Sparkletots Preschool Limited            DS   
3                ST  PCF Sparkletots Preschool Limited            DS   
4                ST  PCF Sparkletots Preschool Limited            DS   

   centre_contact_no   centre_email_address  \
0           68817901   SR.DS.231@pcf.org.sg   
1           6881

This cell prints the first rows of the final combined dataset so you can confirm the merged records, the nested services payload, and the cleaned output before saving.

In [5]:
# Convert the complex map objects into simple text
df_combined['geometry'] = df_combined['geometry'].astype(str)

# Save the combined DataFrame as a JSON file
output_dir = repo_root / "SystemCode/data/processed"
# Safely create the target directory
output_dir.mkdir(parents=True, exist_ok=True)
df_combined_path = output_dir / "kindercompass_master.json"
# Save the combined DataFrame as a JSON file
df_combined.to_json(df_combined_path, orient='records')
# Verify the file was created successfully
if df_combined_path.exists():
    print(f"File saved successfully: {df_combined_path}")
else:
    print("File not found.")

File saved successfully: c:\Users\henry\Documents\NUS\Graduate Certificate in Intelligent Reasoning Systems\practice\KinderCompass\SystemCode\notebooks\poc1\..\..\data\processed\poc1\kindercompass_master.json


This cell verifies that the output JSON file was created successfully, confirming that the full data prep pipeline completed and the master dataset is ready for downstream use.